In [1]:
import sys
from pathlib import Path
import os
import pandas as pd
import numpy as np
import lingam
from sklearn.preprocessing import StandardScaler
import pygraphviz
from causallearn.search.FCMBased import lingam
# Allow running from anywhere by adding project root (one level up from this file) to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.insert(0, project_root)
from features.builder import (
    get_top_concept_ids)

In [2]:
effects_all_df = pd.read_csv("all_feature_effects.csv")

## Furosemide to Other Features

In [3]:
furo_out = (
    effects_all_df
    .query("source == 'exposure_furosemide'")
    .sort_values("pruned_effect", ascending=False)
)
furo_out.head(10)

,rank,source,target,pruned_effect
11,12,exposure_furosemide,Increased_Creatinine,0.090276
357,358,exposure_furosemide,exposure_naproxen,0.000000
594,595,exposure_furosemide,exposure_gentamicin,0.000000
578,579,exposure_furosemide,exposure_tobramycin,0.000000
572,573,exposure_furosemide,exposure_lisinopril,0.000000
538,539,exposure_furosemide,exposure_tacrolimus,0.000000
534,535,exposure_furosemide,exposure_cyclosporine,0.000000
515,516,exposure_furosemide,exposure_losartan,0.000000
479,480,exposure_furosemide,exposure_ceftazidime,0.000000
463,464,exposure_furosemide,exposure_indomethacin,0.000000


## Other Features to Furosemide

In [4]:
to_furo = (
    effects_all_df
    .query("target == 'exposure_furosemide'")
    .sort_values("pruned_effect", ascending=False)
)

to_furo.head(10)


,rank,source,target,pruned_effect
2,3,condition_Essential_hypertension,exposure_furosemide,0.217601
6,7,condition_Acute_kidney_injury,exposure_furosemide,0.160365
15,16,condition_Urinary_tract_infectious_disease,exposure_furosemide,0.086136
16,17,condition_Sepsis,exposure_furosemide,0.079952
17,18,condition_Cirrhosis_of_liver,exposure_furosemide,0.074526
18,19,age_at_outcome,exposure_furosemide,0.065886
22,23,condition_Heart_failure,exposure_furosemide,0.059723
26,27,condition_Hyperkalemia,exposure_furosemide,0.049468
33,34,condition_Chronic_kidney_disease,exposure_furosemide,0.040286
35,36,condition_Diabetes_mellitus,exposure_furosemide,0.039614


## Features to Creatinine

In [8]:
to_creat = (
    effects_all_df
    .query("target == 'Increased_Creatinine'")
    .sort_values("pruned_effect", ascending=False)
)
to_creat = to_creat[to_creat['pruned_effect'] != 0]
to_creat

,rank,source,target,pruned_effect
0,1,age_at_outcome,Increased_Creatinine,0.636470
5,6,condition_Acute_kidney_injury,Increased_Creatinine,0.184286
11,12,exposure_furosemide,Increased_Creatinine,0.090276
47,48,exposure_tacrolimus,Increased_Creatinine,0.029528
51,52,condition_Hyperkalemia,Increased_Creatinine,0.026015
52,53,condition_Sepsis,Increased_Creatinine,0.025832
58,59,condition_Diabetes_mellitus,Increased_Creatinine,0.021008
645,646,exposure_ibuprofen,Increased_Creatinine,-0.024594
646,647,condition_Urinary_tract_infectious_disease,Increased_Creatinine,-0.031453
647,648,condition_Essential_hypertension,Increased_Creatinine,-0.036847


## Feature to Furosemide and Creatinine

In [6]:
rows = []
furo_parents = set(to_furo["source"])
creat_parents = set(to_creat["source"])

shared = sorted(furo_parents & creat_parents)

shared

for feat in shared:
    furo_row = to_furo[to_furo["source"] == feat].iloc[0]
    creat_row = to_creat[to_creat["source"] == feat].iloc[0]

    rows.append({
        "feature": feat,
        "to_furosemide": furo_row["pruned_effect"],
        "to_creatinine": creat_row["pruned_effect"],
        "abs_to_furo": abs(furo_row["pruned_effect"]),
        "abs_to_creat": abs(creat_row["pruned_effect"]),
    })

shared_df = (
    pd.DataFrame(rows)
    .sort_values("abs_to_creat", ascending=False)
    .reset_index(drop=True)
)

shared_df


,feature,to_furosemide,to_creatinine,abs_to_furo,abs_to_creat
0,age_at_outcome,0.065886,0.636470,0.065886,0.636470
1,condition_Acute_kidney_injury,0.160365,0.184286,0.160365,0.184286
2,condition_Essential_hypertension,0.217601,-0.036847,0.217601,0.036847
3,condition_Urinary_tract_infectious_disease,0.086136,-0.031453,0.086136,0.031453
4,exposure_tacrolimus,0.000000,0.029528,0.000000,0.029528
5,condition_Hyperkalemia,0.049468,0.026015,0.049468,0.026015
6,condition_Sepsis,0.079952,0.025832,0.079952,0.025832
7,exposure_ibuprofen,0.000000,-0.024594,0.000000,0.024594
8,condition_Diabetes_mellitus,0.039614,0.021008,0.039614,0.021008


## Features only to Furosemide

In [9]:
to_furo = to_furo[to_furo.pruned_effect != 0]
furo_parents = set(to_furo["source"])
creat_parents = set(to_creat["source"])

furo_only = sorted(furo_parents - creat_parents)

rows = []

for feat in furo_only:
    row = to_furo[to_furo["source"] == feat].iloc[0]
    rows.append({
        "feature": feat,
        "to_furosemide": row["pruned_effect"],
        "abs_to_furo": abs(row["pruned_effect"]),
    })

furo_only_df = (
    pd.DataFrame(rows)
    .sort_values("abs_to_furo", ascending=False)
    .reset_index(drop=True)
)

furo_only_df


,feature,to_furosemide,abs_to_furo
0,condition_Cirrhosis_of_liver,0.074526,0.074526
1,condition_Heart_failure,0.059723,0.059723
2,condition_Chronic_kidney_disease,0.040286,0.040286
